In [1]:
# run_nb("gap_detection.ipynb") で実行する (thesis.ipynb cell 45 の計算部分).
# ギャップの判定条件は thesis_common.ipynb で決める. 実行後 t_ring, r_ring_t, t_gap, r_gap_t, r_in_t, r_out_t, delta_I_t, Kabs_ などが使える
# ======================================================
# 強度プロファイル I_nu(r) から r_disk, r_ring, r_gap の時間進化 (Yamaguchi et al. 2024 の定義)
# I_nu = B_nu(T) (1 - exp(-tau)) , tau = sum_j Sigma_d,j kappa_abs,j   (kappa_abs は optool -dsharp)
# r_disk : F_nu(<r) = 2 pi int_0^r I_nu r' dr' が全フラックス(計算領域内)の 95% となる半径
# r_ring : D(r) = dlnI/dr = 0, D' < 0 (極大) , r_gap : D = 0, D' > 0 (極小)
#   各 r_gap はその外側で最初の r_ring と組にする (論文と同じ). 追加の採用条件はフラグで選ぶ
#   (全て False なら dlnI/dr の符号のみで判定):
#     use_delta_cut : 深さ delta_I = I(r_ring)/I(r_gap) >= delta_min (揺らぎ除去)
#     use_sigma_cut : ガスギャップ Sigma_g/Sigma_g,ini < sigma_ratio_max (r_gap で, Sigma_g,ini は out0)
#     use_rdisk_cut : r_ring <= r_disk
#     use_width_cut : r_ring - r_gap >= w_min_H * H_g(r_gap)  (H_g より狭い凹凸を除く. ビームなしで多い誤検出の対策)
#                     H_g = c_s/Omega , c_s = sqrt(kB T/(mu mH)) , Omega = sqrt(G M_star/r^3) (M_star は input の star_mass)
#   ビームなし (use_beam = False) では use_width_cut と use_delta_cut (delta_min ~ 1.1) の併用を推奨
# ギャップの縁 (論文 Eq. 7, 8): I_edge = 0.5 (I(r_ring) + I(r_gap))  (Case 1)
#   r_out : r_gap < r < r_ring で I = I_edge となる最大の r
#   r_in  : r_peak <= r < r_gap で I = I_edge となる最小の r (r_peak はギャップのすぐ内側の極大, なければ内端)
#   I(r_peak) < I_edge で r_in が決まらない場合は I_edge = 0.5 (I(r_peak) + I(r_gap))  (Case 2)
# use_beam = True : 正面向き円盤を FWHM = beam_fwhm_arcsec * distance_pc [au] のガウスビームで畳み込んでから判定
#   (軸対称なので 2D 畳み込みは I_conv(r) = int I(r') (r'/s^2) exp(-(r^2+r'^2)/2s^2) I0(r r'/s^2) dr' と厳密に書ける)
#   畳み込みは dr_conv 刻みの等間隔格子 (r <= r_conv_max) 上で行い, 計算領域の内側 (r < r_in) は I = 0
# file 列: out (5行目以降) 0:r_cen 1~:Sigma_d,j   out_gas 1:Sigma_g 12:T
# 関数 (find_radii, crossing, Planck_Func_nu, calc_Kabs, calc_beam) は thesis_common.ipynb で定義する
# ======================================================

os.environ["PATH"] = os.path.expanduser("~/bin") + ":" + os.environ["PATH"]

AU = 1.495978707e13          # cm
h = 6.62607015e-27           # erg s
light_speed = 2.99792458e10  # cm s^-1
kB = 1.380649e-16            # erg K^-1
G = 6.67408e-08              # cm^3 g^-1 s^-2
Msolar = 1.989e33            # g
mH = 1.67353e-24             # g
mu = 2.34

outdir = BASE_DIR
resultdir = MODEL
t_start = 0

path = outdir + resultdir
years = list_years(path, t_start)

# ---- 吸収係数 (各ダストビン, optool -dsharp) ----
dradius = np.array([float(i) for i in linecache.getline(path + "/out{}".format(years[0]), 3).split()])
linecache.clearcache()
Kabs_ = calc_Kabs(dradius)

sigmag_ini = np.loadtxt(path + "/out0_gas", usecols=(1,))
M_star = Msolar
with open(path + "/input") as f:
    for line in f:
        if line.split("//")[0].split("=")[0].strip() == "star_mass":
            M_star = float(line.split("//")[0].split("=")[1]) * Msolar

beam_fwhm_au = beam_fwhm_arcsec * distance_pc   # 1" at d pc = d au
if use_beam:
    r_u, W = calc_beam()

t_disk, r_disk_t, t_ring, r_ring_t, t_gap, r_gap_t, r_in_t, r_out_t, delta_I_t = [], [], [], [], [], [], [], [], []
for year in years:
    d = np.loadtxt(path + "/out{}".format(year), skiprows=5)
    gas = np.loadtxt(path + "/out{}_gas".format(year), usecols=(1, 12))
    sigmag, T = gas[:, 0], gas[:, 1]
    r = d[:, 0]
    sigmad = np.where(np.abs(d[:, 1:]) > 1.0e-50, d[:, 1:], 0.0)
    tau = sigmad @ Kabs_
    I = Planck_Func_nu(T, Lambda_obs) * (1.0 - np.exp(-tau)) * 1.0e23   # Jy/sr
    Hg = np.sqrt(kB * T / (mu * mH)) / np.sqrt(G * M_star / r**3)

    if use_beam:
        I_u = np.interp(r_u, r, I, left=0.0, right=0.0)
        r_disk, rings, gaps, edges_in, edges_out, depths = find_radii(r_u, W @ I_u, np.interp(r_u, r, sigmag / sigmag_ini), np.interp(r_u, r, Hg))
    else:
        r_disk, rings, gaps, edges_in, edges_out, depths = find_radii(r, I, sigmag / sigmag_ini, Hg)
    t_disk.append(year)
    r_disk_t.append(r_disk)
    t_ring += [year] * len(rings)
    r_ring_t += rings
    t_gap += [year] * len(gaps)
    r_gap_t += gaps
    r_in_t += edges_in
    r_out_t += edges_out
    delta_I_t += depths

t_disk = np.array(t_disk) / 1.0e3
beam_label = r", beam {}$^{{\prime\prime}}$ @ {:.0f} pc ({:.0f} au)".format(beam_fwhm_arcsec, distance_pc, beam_fwhm_au) if use_beam else ", no beam"

# r_in_t, r_out_t は analysis_obs_theory.ipynb の縁のセルが上書きするので, I_dust の結果を別名でも残す [t: kyr, r: cm]
res_I = {"t": np.array(t_gap) / 1.0e3, "ring": np.array(r_ring_t), "gap": np.array(r_gap_t), "in": np.array(r_in_t), "out": np.array(r_out_t)}


NameError: name 'os' is not defined